# 산타 인형 검출 YOLO11n 학습 (홈키파 거실 노드)

1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 셀을 위에서부터 차례로 실행 (▶)
3. 마지막 셀에서 `santa_yolo11n.onnx`가 내려받아지면 끝

In [ ]:
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

## 1. 데이터 올리기
Roboflow에서 **YOLOv11(또는 YOLOv8) 형식 zip**으로 내보낸 파일을 올린다. 실행하면 파일 선택 창이 뜬다.

In [ ]:
from google.colab import files
import zipfile, glob, os
up = files.upload()
zname = list(up)[0]
zipfile.ZipFile(zname).extractall('data')
yaml = glob.glob('data/**/data.yaml', recursive=True)[0]
print('data.yaml:', yaml)
print(open(yaml).read())

## 2. 학습
이미지 크기는 거실 노드와 같은 320. 사진이 적으면 epochs를 늘린다(최대 150).

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data=yaml, imgsz=320, epochs=80, batch=32, patience=20,
            fliplr=0.5, degrees=5, hsv_v=0.4, name='santa')

## 3. 결과 확인
`mAP50`이 0.9 이상이면 충분하다. 검증 사진에 상자가 제대로 쳐졌는지 눈으로도 본다.

In [ ]:
best = 'runs/detect/santa/weights/best.pt'
m = YOLO(best)
metrics = m.val(data=yaml, imgsz=320)
print('mAP50:', round(metrics.box.map50, 3))
from IPython.display import Image, display
display(Image('runs/detect/santa/val_batch0_pred.jpg', width=640))

## 4. ONNX로 내보내기 (거실 노드용)

In [ ]:
onnx = YOLO(best).export(format='onnx', imgsz=320, opset=12, simplify=True)
os.rename(onnx, 'santa_yolo11n.onnx')
files.download('santa_yolo11n.onnx')